# CIFAR-10 Federated Learning (Simplified)

1. CIFAR-10 로드
2. 2개 클라이언트로 분할
3. 각 클라이언트가 로컬 CNN 학습
4. FedAvg로 모델 취합
5. 전체 테스트 정확도 확인

In [1]:
import copy
from collections import OrderedDict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLIENTS = 2
NUM_ROUNDS = 2
LOCAL_EPOCHS = 1
BATCH_SIZE = 64
LR = 0.001
TRAIN_SIZE = 4000
TEST_SIZE = 1000

## 1. 데이터 로드

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_full = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_full = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_data = Subset(train_full, range(TRAIN_SIZE))
test_data = Subset(test_full, range(TEST_SIZE))
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

100%|██████████| 170M/170M [1:04:08<00:00, 44.3kB/s] 


## 2. 클라이언트로 데이터 분할 (단순 균등 분할)

In [3]:
client_size = len(train_data) // NUM_CLIENTS
client_loaders = []
for i in range(NUM_CLIENTS):
    subset = Subset(train_data, range(i*client_size, (i+1)*client_size))
    client_loaders.append(DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True))
print(f"클라이언트 {NUM_CLIENTS}개, 각 {client_size}개 샘플")

클라이언트 2개, 각 2000개 샘플


## 3. 간단한 CNN 모델

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Linear(32 * 8 * 8, num_classes)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

global_model = SimpleCNN().to(device)

## 4. 학습 / 평가 / FedAvg 함수

In [5]:
def train_client(model, loader):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
    return loss.item()

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

def fedavg(state_dicts):
    avg = OrderedDict()
    for key in state_dicts[0]:
        avg[key] = sum(sd[key] for sd in state_dicts) / len(state_dicts)
    return avg

## 5. 연합학습 실행

In [6]:
for round_idx in range(NUM_ROUNDS):
    local_states = []
    for loader in client_loaders:
        local_model = copy.deepcopy(global_model)
        train_client(local_model, loader)
        local_states.append(local_model.state_dict())

    global_model.load_state_dict(fedavg(local_states))
    acc = evaluate(global_model, test_loader)
    print(f"Round {round_idx+1}: Test Accuracy = {acc:.2f}%")

Round 1: Test Accuracy = 33.90%
Round 2: Test Accuracy = 38.30%


## 요약

- CIFAR-10(10종 컬러 이미지)에 연합학습 구조를 적용
- 데이터를 클라이언트 2곳으로 나눠 각각 로컬 학습
- FedAvg로 두 모델의 가중치를 평균내어 글로벌 모델 생성
- 개인 데이터는 로컬에 유지하고 모델 업데이트만 공유한다는 점이 핵심